# GMTI-Net — NTIRE 2026 Local Training & Evaluation Notebook
**Complete, production-ready notebook:** setup, tests, staged training, validation, checkpoint averaging, self-ensemble & multiscale inference, visualizations, and 

## Table of Contents
0. Title & TOC (this cell)
1–2. Configuration (paths, flags, stage iters)
3. Mount Drive
4. Repo setup (clone/zip/drive)
5. GPU & RAM check
6. Install dependencies
7. Dataset preparation (NTIRE detection + synthetic fallback)
8. Smoke forward pass (sanity)
9. Run test suite (pytest)
10. Stage 1 training — pretrain (256px)
11. Stage 2 training — mid (384px)
12. Stage 3 training — fine-tune (512px)
13. Post-training validation (best EMA)
14. Checkpoint averaging (avg_ema.pth)
15. Inference & self-ensemble (save results + PSNR)
16. Visualizations (sample predictions, flows, occlusion)
17. Drive backup
18. TensorBoard (launch)
19. Hyperparameter Sweep (Extra 1)
20. Automatic Model Comparison (Extra 2)
21. Run summary & next steps

Run this notebook top-to-bottom. Edit the config cell before doing heavy work.


In [1]:
# Cell 1 — Deterministic Seed
SEED = 2026
import random
import numpy as np
import torch

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
print(f'Seed set to {SEED} for reproducibility.')


Seed set to 2026 for reproducibility.


In [2]:
# Cell 1 — Configuration (edit these before running heavy cells)
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3' # Suppress benign TensorFlow/CUDA log noise
from pathlib import Path

# -------------- Basic repo settings --------------
# Determine repo ROOT assuming this notebook is in docs/ or root/
cwd = Path(os.path.abspath(''))
REPO_DIR = cwd.parent if cwd.name == 'docs' else cwd

# -------------- Quick run / debug --------------
QUICK_RUN = True       # False = Run the massive 240k iteration NTIRE pipeline
DEBUG = False           # True = save visualizations frequently

# -------------- Staged training iterations (adjust for resources) --------------
STAGE1_ITERS = 120_000  # 256px pretrain (set to 120k on full runs)
STAGE2_ITERS = 80_000   # 384px mid-stage
STAGE3_ITERS = 40_000   # 512px fine-tune

# Quick-run overrides
if QUICK_RUN:
    STAGE1_ITERS = 20
    STAGE2_ITERS = 10
    STAGE3_ITERS = 10

# -------------- Paths --------------
# Local paths
CHECKPOINT_DIR = REPO_DIR / 'checkpoints'
LOG_DIR = REPO_DIR / 'logs'
RESULTS_DIR = REPO_DIR / 'results'
DATA_DIR = REPO_DIR / 'data'
NTIRE_TRAIN_DIR = DATA_DIR / 'NTIRE/train'
NTIRE_VAL_DIR = DATA_DIR / 'NTIRE/val'

# -------------- Training config file --------------
CONFIG_YAML = REPO_DIR / 'config.yaml'

# -------------- Experiment metadata --------------
EXPERIMENT_NAME = 'gmti_ntire_run'
RUN_DIR = LOG_DIR / EXPERIMENT_NAME

# -------------- Other misc --------------
DEVICE = 'cuda' if (os.getenv('CUDA_VISIBLE_DEVICES') is not None or os.path.exists('/dev/nvidia0') or (hasattr(os, "name") and os.name == "nt" and os.system("nvidia-smi >nul 2>&1") == 0)) else 'cpu'
print(f'CONFIG: REPO_DIR={REPO_DIR}, QUICK_RUN={QUICK_RUN}, DEVICE={DEVICE}')


CONFIG: REPO_DIR=C:\Visual Studio Code\SEM - 6\IVP\Boom, QUICK_RUN=True, DEVICE=cuda


## Quick hyperparameter decisions (edit config.yaml for final changes)
- `proj_dim`: **160** (recommended final)
- `corr_temp`: **0.08** (sweep: 0.05, 0.075, 0.1, 0.15)
- `laplacian weight`: **0.3**
- `loss.mse`: **0.1** (direct MSE to bias for PSNR)
- `ema.decay`: **0.99995**
- `multiscale weights`: **[0.05, 0.2, 0.7, 1.0]**
- `dataset mixing`: stage-dependent (pretrain: vimeo heavy; finetune: ntire only)
- `inference`: average top-K validation ckpts + self-ensemble flips + multiscale


In [3]:
# Cell 4 — Repo setup: Add to python path
import sys, os
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
os.chdir(str(REPO_DIR))
print('Working directory:', os.getcwd())


Working directory: C:\Visual Studio Code\SEM - 6\IVP\Boom


In [4]:
# Cell 5 — GPU & RAM check
import torch, psutil, shutil, sys
print('Python:', sys.version.splitlines()[0])
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('CUDA device name:', torch.cuda.get_device_name(0))
    prop = torch.cuda.get_device_properties(0)
    print(f'Total VRAM: {prop.total_memory / (1024**3):.2f} GB')
    print(f'cuDNN version: {torch.backends.cudnn.version()}')
print('System RAM:', psutil.virtual_memory().total / (1024**3), 'GB')
print('Disk free:', shutil.disk_usage('/').free / (1024**3), 'GB')
print('REPO_DIR exists:', REPO_DIR.exists())


Python: 3.11.6 (tags/v3.11.6:8b6ee5b, Oct  2 2023, 14:57:12) [MSC v.1935 64 bit (AMD64)]
Torch: 2.0.1+cu117


CUDA available: True
CUDA device name: NVIDIA GeForce RTX 4060 Laptop GPU
Total VRAM: 8.00 GB
cuDNN version: 8500
System RAM: 13.700675964355469 GB
Disk free: 178.16080474853516 GB
REPO_DIR exists: True


In [5]:
# Cell 6 — Install dependencies (runs pip). Modify as needed.
import subprocess, sys, os
req = REPO_DIR / 'requirements.txt'
if req.exists():
    print('Installing from', req)
    subprocess.run(f'pip install -r {req}', shell=True, check=False)
else:
    print('requirements.txt not found in repo. Install dependencies manually.')
# Additional optional libraries for evaluation/visualization:
subprocess.run('pip install lpips==0.1.4 pytorch-msssim==0.3.7 scikit-image psutil', shell=True, check=False)
print('Dependencies installed.')


Installing from C:\Visual Studio Code\SEM - 6\IVP\Boom\requirements.txt


Dependencies installed.


In [6]:
# Cell 7 — Dataset preparation
import os, random
from PIL import Image
import numpy as np

# Create directories
for d in [CHECKPOINT_DIR, LOG_DIR, RESULTS_DIR, DATA_DIR]:
    d.mkdir(parents=True, exist_ok=True)

def has_ntire_data():
    return NTIRE_TRAIN_DIR.exists() and any(NTIRE_TRAIN_DIR.glob('**/*'))

def download_vimeo90k():
    target_dir = DATA_DIR
    target_dir.mkdir(parents=True, exist_ok=True)

    vimeo_dir = target_dir / 'vimeo_triplet'
    if vimeo_dir.exists() and (vimeo_dir / 'tri_trainlist.txt').exists():
        print(f'Vimeo90K already exists in {vimeo_dir}')
        return

    print(f'Downloading Vimeo90K triplet dataset (33GB) to {target_dir}... This will take a while.')
    import urllib.request
    from tqdm import tqdm

    class DownloadProgressBar(tqdm):
        def update_to(self, b=1, bsize=1, tsize=None):
            if tsize is not None:
                self.total = tsize
            self.update(b * bsize - self.n)

    vimeo_zip = target_dir / 'vimeo_triplet.zip'
    url = 'http://data.csail.mit.edu/tofu/dataset/vimeo_triplet.zip'
    with DownloadProgressBar(unit='B', unit_scale=True, miniters=1, desc=url.split('/')[-1]) as t:
        urllib.request.urlretrieve(url, filename=vimeo_zip, reporthook=t.update_to)

    print('\nUnzipping Vimeo90K...')
    import subprocess
    subprocess.run(f'unzip -q {vimeo_zip} -d {target_dir}', shell=True)
    vimeo_zip.unlink(missing_ok=True) # clean up zip to save space
    print('Vimeo90K downloaded and unzipped successfully.')

if QUICK_RUN:
    print('QUICK_RUN is True. Generating tiny synthetic NTIRE dataset for smoke testing.')
    if not has_ntire_data():
        for split, num_frames in [('train', 20), ('val', 10)]:
            vid_dir = DATA_DIR / 'NTIRE' / split / 'vid_1'
            vid_dir.mkdir(parents=True, exist_ok=True)
            for i in range(num_frames):
                base = np.zeros((256, 256, 3), dtype=np.uint8)
                x = 30 + i * 2
                x = min(x, 256 - 32)
                base[64:192, x:x+32, :] = 255
                Image.fromarray(base).save(vid_dir / f'{i:04d}.png')
        print('Synthetic dataset created.')

        NTIRE_TRAIN_DIR = DATA_DIR / 'NTIRE' / 'train'
        NTIRE_VAL_DIR = DATA_DIR / 'NTIRE' / 'val'
else:
    print('Performing full dataset download check...')
    download_vimeo90k()

    if not has_ntire_data():
        print('WARNING: NTIRE data not found at', NTIRE_TRAIN_DIR)
        print('Please manually upload the NTIRE dataset to', NTIRE_TRAIN_DIR)
    else:
        print('NTIRE data found at', NTIRE_TRAIN_DIR)


QUICK_RUN is True. Generating tiny synthetic NTIRE dataset for smoke testing.


In [7]:
# Cell 8 — Smoke test: import model and run forward with random tensors
import sys, importlib
sys.path.append(str(REPO_DIR))

try:
    from models.gmti_net import GMTINet
    print('Imported GMTINet')
    m = GMTINet().to(DEVICE).eval()
    import torch
    with torch.no_grad():
        L = torch.rand(1, 3, 128, 128).to(DEVICE)
        R = torch.rand(1, 3, 128, 128).to(DEVICE)
        pred = m.inference(L, R)
        print('Output shape:', pred.shape)
    n_params = sum(p.numel() for p in m.parameters() if p.requires_grad)
    print(f'Model parameters: {n_params/1e6:.2f} M')
    del m; torch.cuda.empty_cache()
    print('Smoke test PASSED')
except Exception as e:
    print('Smoke test failed: could not instantiate/forward model:', e)
    raise


Imported GMTINet


Output shape: torch.Size([1, 3, 128, 128])
Model parameters: 6.41 M
Smoke test PASSED


In [8]:
# Cell 9 — Run unit tests
import subprocess, sys, os
print('Running pytest on tests/ ...')
res = subprocess.run(
    f'PYTHONPATH={REPO_DIR} pytest {REPO_DIR}/tests -q --tb=short',
    shell=True, capture_output=True, text=True
)
print(res.stdout or '(no stdout)')
if res.stderr:
    print('STDERR:', res.stderr[-2000:])
print('pytest exit code:', res.returncode)
if res.returncode != 0:
    print('Some tests failed — fix them before full training.')
else:
    print('All tests passed!')


Running pytest on tests/ ...
(no stdout)
STDERR: 'PYTHONPATH' is not recognized as an internal or external command,
operable program or batch file.

pytest exit code: 1
Some tests failed — fix them before full training.


In [9]:
# Cell 10 — Stage 1 training (256 px pretrain)
import subprocess, shlex, os, time

if QUICK_RUN:
    print('QUICK_RUN enabled — running a tiny 20-iter test with forced ckpt saves.')
    cmd = (f'python {REPO_DIR}/train.py --config {CONFIG_YAML} '
           f'--stage 1 --patch_size 256 --max_iters 20 --save_freq 10')
else:
    cmd = (f'python {REPO_DIR}/train.py --config {CONFIG_YAML} '
           f'--stage 1 --patch_size 256 --max_iters {STAGE1_ITERS}')
print('Running Stage 1 train:', cmd)
p = subprocess.Popen(shlex.split(cmd), stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
try:
    for line in p.stdout:
        print(line, end='', flush=True)
except KeyboardInterrupt:
    p.terminate()
    raise
rc = p.wait()
print('Stage 1 training finished with return code', rc)


QUICK_RUN enabled — running a tiny 20-iter test with forced ckpt saves.
Running Stage 1 train: python C:\Visual Studio Code\SEM - 6\IVP\Boom/train.py --config C:\Visual Studio Code\SEM - 6\IVP\Boom\config.yaml --stage 1 --patch_size 256 --max_iters 20 --save_freq 10


python: can't open file 'C:\\Visual Studio Code\\SEM - 6\\IVP\\Boom\\Visual': [Errno 2] No such file or directory


Stage 1 training finished with return code 2


In [10]:
# Cell 11 — Stage 2 training (384 px mid-stage)
import subprocess, shlex
if QUICK_RUN:
    print('QUICK_RUN enabled — skipping Stage 2.')
else:
    cmd = (f'python {REPO_DIR}/train.py --config {CONFIG_YAML} '
           f'--stage 2 --patch_size 384 --max_iters {STAGE2_ITERS} '
           f'--resume {CHECKPOINT_DIR}/latest.pth')
    print('Running Stage 2 train:', cmd)
    p = subprocess.Popen(shlex.split(cmd), stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    try:
        for line in p.stdout:
            print(line, end='', flush=True)
    except KeyboardInterrupt:
        p.terminate()
        raise
    rc = p.wait()
    print('Stage 2 finished with return code', rc)


QUICK_RUN enabled — skipping Stage 2.


In [11]:
# Cell 12 — Stage 3 fine-tune (512 px)
import subprocess, shlex
if QUICK_RUN:
    print('QUICK_RUN enabled — skipping Stage 3.')
else:
    cmd = (f'python {REPO_DIR}/train.py --config {CONFIG_YAML} '
           f'--stage 3 --patch_size 512 --max_iters {STAGE3_ITERS} '
           f'--resume {CHECKPOINT_DIR}/latest.pth')
    print('Running Stage 3 train:', cmd)
    p = subprocess.Popen(shlex.split(cmd), stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    try:
        for line in p.stdout:
            print(line, end='', flush=True)
    except KeyboardInterrupt:
        p.terminate()
        raise
    rc = p.wait()
    print('Stage 3 finished with return code', rc)


QUICK_RUN enabled — skipping Stage 3.


In [12]:
# Cell 14 — Validation: load best EMA if present otherwise latest.pth
import torch, os, subprocess
from utils.io import extract_model_state
ckpt_candidates = (
    list(CHECKPOINT_DIR.glob('best_ema_*.pth')) +
    list(CHECKPOINT_DIR.glob('best_ema*.pth')) +
    list(CHECKPOINT_DIR.glob('avg_ema.pth')) +
    list(CHECKPOINT_DIR.glob('latest.pth'))
)
if len(ckpt_candidates) == 0:
    raise RuntimeError('No checkpoints found in ' + str(CHECKPOINT_DIR))
ckpt_path = ckpt_candidates[0]
print('Validating checkpoint:', ckpt_path)

# Stream validate.py output live
cmd = f'python {REPO_DIR}/validate.py --config {CONFIG_YAML} --checkpoint {ckpt_path} --outdir {RESULTS_DIR}/val'
print('Running:', cmd)
import subprocess, shlex
proc = subprocess.Popen(shlex.split(cmd), stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in proc.stdout:
    print(line, end='', flush=True)
proc.wait()
print('Validation exit code:', proc.returncode)


Validating checkpoint: C:\Visual Studio Code\SEM - 6\IVP\Boom\checkpoints\latest.pth
Running: python C:\Visual Studio Code\SEM - 6\IVP\Boom/validate.py --config C:\Visual Studio Code\SEM - 6\IVP\Boom\config.yaml --checkpoint C:\Visual Studio Code\SEM - 6\IVP\Boom\checkpoints\latest.pth --outdir C:\Visual Studio Code\SEM - 6\IVP\Boom\results/val


python: can't open file 'C:\\Visual Studio Code\\SEM - 6\\IVP\\Boom\\Visual': [Errno 2] No such file or directory


Validation exit code: 2


In [13]:
# Cell 15 — Average top-5 validation PSNR checkpoints
import torch, glob, re

def average_top_checkpoints(ckpt_dir, pattern='best_ema_*.pth', n=5):
    from pathlib import Path
    files = list(Path(ckpt_dir).glob(pattern))
    if not files:
        # fallback to iter_*
        files = list(Path(ckpt_dir).glob('iter_*.pth'))
        if not files:
            raise RuntimeError('No checkpoint files found for averaging')
        # sort by iteration
        extracted = []
        for f in files:
            match = re.search(r'iter_(\d+)\.pth', f.name)
            val = float(match.group(1)) if match else 0.0
            extracted.append((val, f))
    else:
        # valid best_emas, sort by PSNR
        extracted = []
        for f in files:
            match = re.search(r'best_ema_(\d+\.\d+)\.pth', f.name)
            val = float(match.group(1)) if match else 0.0
            extracted.append((val, f))
    
    extracted.sort(key=lambda x: x[0], reverse=True) # Top N
    top_files = [f for _, f in extracted[:n]]
    print(f'Averaging top {len(top_files)} checkpoints:')
    for v, f in extracted[:n]: print(f'  PSNR/Iter: {v:.3f} -> {f.name}')

    avg_state = None
    for f in top_files:
        ckpt = torch.load(f, map_location='cpu', weights_only=False)
        st = ckpt.get('ema', ckpt.get('model', ckpt.get('state_dict')))
        if avg_state is None:
            avg_state = {k: v.clone().float() for k, v in st.items()}
        else:
            for k in avg_state:
                avg_state[k] += st[k].float()
    for k in avg_state:
        avg_state[k] /= len(top_files)
    avg_ckpt = {'ema': avg_state}
    out_path = Path(ckpt_dir) / 'avg_ema.pth'
    torch.save(avg_ckpt, out_path)
    print('Saved averaged checkpoint to', out_path)
    return out_path

avg_ckpt_path = average_top_checkpoints(CHECKPOINT_DIR, 'best_ema_*.pth', n=5)


Averaging top 2 checkpoints:
  PSNR/Iter: 2.000 -> iter_2.pth
  PSNR/Iter: 1.000 -> iter_1.pth


Saved averaged checkpoint to C:\Visual Studio Code\SEM - 6\IVP\Boom\checkpoints\avg_ema.pth


In [14]:
# Cell 16 — Inference with self-ensemble and multiscale
import torch, os, glob, math
from PIL import Image
import numpy as np
from torchvision.transforms.functional import to_tensor, to_pil_image
import torchvision.transforms.functional as TF

sys.path.append(str(REPO_DIR))
from inference import load_model_for_inference
model, cfg = load_model_for_inference(avg_ckpt_path, device=DEVICE)

def run_self_ensemble(model, L_img, R_img, scales=[1.0, 1.25], flips=['none', 'h', 'v', 'hv']):
    """
    L_img, R_img: PIL images or tensors [C,H,W] in [0,1] float.
    Returns averaged prediction tensor [C,H,W].
    """
    model.eval()
    preds = []
    Ls0 = to_tensor(L_img) if not isinstance(L_img, torch.Tensor) else L_img
    Rs0 = to_tensor(R_img) if not isinstance(R_img, torch.Tensor) else R_img
    orig_H, orig_W = Ls0.shape[-2], Ls0.shape[-1]
    for s in scales:
        Ls = Ls0.clone(); Rs = Rs0.clone()
        if s != 1.0:
            H = int(Ls.shape[1] * s); W = int(Ls.shape[2] * s)
            Ls = TF.resize(Ls, (H, W))
            Rs = TF.resize(Rs, (H, W))
        for f in flips:
            Lt, Rt = Ls.clone(), Rs.clone()
            if f == 'h':  Lt = Lt.flip(-1);         Rt = Rt.flip(-1)
            if f == 'v':  Lt = Lt.flip(-2);         Rt = Rt.flip(-2)
            if f == 'hv': Lt = Lt.flip(-1).flip(-2); Rt = Rt.flip(-1).flip(-2)
            with torch.no_grad():
                out, aux = model(Lt.unsqueeze(0).to(DEVICE), Rt.unsqueeze(0).to(DEVICE))
                out = out.squeeze(0).cpu()
                # Extract flow for debug visualizations (Extra request)
                if f == 'none' and s == 1.0:
                    flow_LM = aux.get('flow_forward')
                    if flow_LM is not None: global_flow = flow_LM[-1].detach().cpu().squeeze(0)
            # inverse flip
            if f == 'h':  out = out.flip(-1)
            if f == 'v':  out = out.flip(-2)
            if f == 'hv': out = out.flip(-1).flip(-2)
            # resize back if scaled
            if s != 1.0:
                out = TF.resize(out, (orig_H, orig_W))
            preds.append(out)
    return torch.stack(preds).mean(0), global_flow

def compute_psnr(pred, gt):
    mse = torch.mean((pred - gt) ** 2).item()
    if mse < 1e-10: return 100.0
    return -10.0 * math.log10(mse)

# Run on NTIRE val frames
val_images = sorted(NTIRE_VAL_DIR.glob('*_M.png'))[:5] if NTIRE_VAL_DIR.exists() else []
out_dir = RESULTS_DIR / 'inference'
diag_dir = RESULTS_DIR / 'diagnostics'
out_dir.mkdir(parents=True, exist_ok=True)
diag_dir.mkdir(parents=True, exist_ok=True)
psnr_vals = []

for p in val_images:
    Lp = p.parent / (p.stem.replace('_M', '_L') + p.suffix)
    Rp = p.parent / (p.stem.replace('_M', '_R') + p.suffix)
    if not Lp.exists() or not Rp.exists(): continue
    Limg = Image.open(Lp).convert('RGB')
    Rimg = Image.open(Rp).convert('RGB')
    gt   = to_tensor(Image.open(p).convert('RGB'))
    pred, flow_LM = run_self_ensemble(model, Limg, Rimg, scales=[1.0, 1.25], flips=['none','h','v','hv'])
    pred = pred.clamp(0.0, 1.0)
    psnr = compute_psnr(pred, gt)
    psnr_vals.append(psnr)
    out_path = out_dir / f'pred_{p.stem}.png'
    to_pil_image(pred).save(out_path)
    if flow_LM is not None:
        np.save(diag_dir / f'{p.stem}_flow.npy', flow_LM.numpy())
    print(f'Saved {out_path}  PSNR: {psnr:.4f} dB')

if psnr_vals:
    print(f'Mean PSNR on sample: {sum(psnr_vals)/len(psnr_vals):.4f} dB')
else:
    print('No val images found. Place NTIRE val frames in', NTIRE_VAL_DIR)


[load_model_for_inference] Loaded C:\Visual Studio Code\SEM - 6\IVP\Boom\checkpoints\avg_ema.pth  (6.41 M params)  device=cuda
No val images found. Place NTIRE val frames in C:\Visual Studio Code\SEM - 6\IVP\Boom\data\NTIRE\val


In [15]:
# Cell 17 — Visualizations: show predictions, flows, magnitude, and error heatmaps
import matplotlib.pyplot as plt
import numpy as np
import torch.nn.functional as F
from PIL import Image
from torchvision.transforms.functional import to_tensor
from IPython.display import display

def flow_to_image(flow):
    """Visualise optical flow using HSV color wheel. flow: [2,H,W] tensor."""
    import cv2
    fx = flow[0].flatten(); fy = flow[1].flatten()
    mag, ang = cv2.cartToPolar(fx, fy)
    mag, ang = mag.reshape(flow.shape[1:]), ang.reshape(flow.shape[1:])
    hsv = np.zeros((*flow.shape[1:], 3), dtype=np.float32)
    hsv[..., 0] = ang / (2 * np.pi)
    hsv[..., 1] = np.clip(mag / (np.max(mag) + 1e-6), 0, 1)
    hsv[..., 2] = (hsv[..., 1] > 0).astype(np.float32)
    rgb = cv2.cvtColor((hsv * 255).astype(np.uint8), cv2.COLOR_HSV2RGB)
    return rgb

def flow_backward_warp(img, flow):
    B, C, H, W = img.shape
    grid_x, grid_y = torch.meshgrid(torch.arange(W), torch.arange(H), indexing='xy')
    grid_x = grid_x.float().to(img.device).unsqueeze(0)
    grid_y = grid_y.float().to(img.device).unsqueeze(0)
    u, v = flow[:, 0], flow[:, 1]
    x = grid_x + u
    y = grid_y + v
    x = 2 * (x / (W - 1)) - 1
    y = 2 * (y / (H - 1)) - 1
    grid = torch.stack((x, y), dim=-1)
    warped = F.grid_sample(img, grid, align_corners=True, mode='bilinear', padding_mode='reflection')
    return warped

# Show sample predictions from the inference step
sample_fnames = sorted((RESULTS_DIR / 'inference').glob('pred_*.png'))[:3]
if not sample_fnames:
    print('No predictions found in results/inference/ — run Cell 15 first.')
for f in sample_fnames:
    print('Sample:', f.name)
    img = Image.open(f).convert('RGB')
    display(img)
    # show flow / occlusion if saved as .npy diagnostics
    base = f.stem.replace('pred_', '')
    flow_file = RESULTS_DIR / 'diagnostics' / f'{base}_flow.npy'
    occ_file  = RESULTS_DIR / 'diagnostics' / f'{base}_occ.npy'
    import torch
    if flow_file.exists():
        flow = torch.from_numpy(np.load(flow_file))  # [2,H,W]
        plt.figure(figsize=(15, 5))
        plt.subplot(1, 4, 1)
        plt.imshow(flow_to_image(flow))
        plt.title('Flow HSV'); plt.axis('off')
        
        plt.subplot(1, 4, 2)
        mag = np.sqrt(flow[0]**2 + flow[1]**2)
        plt.imshow(mag, cmap='magma')
        plt.title('Flow Magnitude'); plt.axis('off'); plt.colorbar(fraction=0.046, pad=0.04)
        
        # Error Map (Improvement/Extra 3)
        # Load L and GT to measure tracking error
        Lp = NTIRE_VAL_DIR / f'{base}_L.png'
        Mp = NTIRE_VAL_DIR / f'{base}_M.png'
        if Lp.exists() and Mp.exists():
            L_t = to_tensor(Image.open(Lp)).unsqueeze(0)
            M_t = to_tensor(Image.open(Mp)).unsqueeze(0)
            warped_L = flow_backward_warp(L_t, flow.unsqueeze(0))
            err_map = torch.abs(warped_L - M_t).squeeze(0).mean(0).numpy()
            
            plt.subplot(1, 4, 3)
            plt.imshow(err_map, cmap='hot')
            plt.title('Warp Error |warp(L,F)-GT|'); plt.axis('off'); plt.colorbar(fraction=0.046, pad=0.04)
            
            # Prediction Error
            pred_err = torch.abs(to_tensor(img) - M_t.squeeze(0)).mean(0).numpy()
            plt.subplot(1, 4, 4)
            plt.imshow(pred_err, cmap='hot')
            plt.title('Pred Error |Pred-GT|'); plt.axis('off'); plt.colorbar(fraction=0.046, pad=0.04)
        plt.tight_layout(); plt.show()


No predictions found in results/inference/ — run Cell 15 first.


## Cell 18 — Launch TensorBoard
Run the cell below to launch TensorBoard locally.


In [16]:
# Launch tensorboard (Local)
import os, subprocess
print('Log dir:', LOG_DIR)
try:
    print('Start tensorboard in a separate terminal using: tensorboard --logdir logs')
    # %load_ext tensorboard
    # %tensorboard --logdir {LOG_DIR}
except Exception as e:
    print('Could not start TensorBoard:', e)


Log dir: C:\Visual Studio Code\SEM - 6\IVP\Boom\logs
Start tensorboard in a separate terminal using: tensorboard --logdir logs


In [17]:
# Cell 20 — Automated Hyperparameter Sweep (Optional Extra)
import yaml
import subprocess
import itertools
print('Running Automated Hyperparameter Sweep...')
if QUICK_RUN:
    print('Skipping sweep during QUICK_RUN.')
else:
    sweep_grid = {
        'corr_temp': [0.05, 0.075, 0.1],
        'laplacian': [0.25, 0.3, 0.35]
    }
    print('Grid:', sweep_grid)
    # Uncomment to actually run full stage 1 for each combo and record best PSNR
    '''
    keys = list(sweep_grid.keys())
    combos = list(itertools.product(*(sweep_grid[k] for k in keys)))
    for combo in combos:
        print(f'\n>> Testing {dict(zip(keys, combo))}')
        with open(CONFIG_YAML, 'r') as f:
            cfg = yaml.safe_load(f)
        cfg['correlation']['corr_temp'] = combo[0]
        cfg['loss']['laplacian'] = combo[1]
        with open(CONFIG_YAML, 'w') as f:
            yaml.safe_dump(cfg, f)
        # run training subprocess for say 5k iterations, validate, log score
    '''


Running Automated Hyperparameter Sweep...
Skipping sweep during QUICK_RUN.


In [18]:
# Cell 21 — Automatic Model Comparison (Plotting Multiple Runs, Optional Extra)
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

print('Plotting PSNR curves from multiple runs...')
# Looks for CSV logs exported by validate.py or tensorboard events
val_summaries = list(RESULTS_DIR.glob('*/val_summary.csv'))
if len(val_summaries) > 0:
    plt.figure(figsize=(8, 5))
    for summary_path in val_summaries:
        df = pd.read_csv(summary_path)
        run_name = summary_path.parent.name
        if 'PSNR' in df:
            plt.plot(df['frame'].values, df['PSNR'].values, marker='o', label=run_name)
    plt.title('Validation PSNR Comparison')
    plt.ylabel('PSNR (dB)')
    plt.xlabel('Frame')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()
else:
    print('No multiple val_summary.csv runs found to plot. Run evaluation for different configs and save them in separate folders.')


Plotting PSNR curves from multiple runs...
No multiple val_summary.csv runs found to plot. Run evaluation for different configs and save them in separate folders.


## Cell 19 — Run summary & next steps

**What we executed**
- Repo & environment setup
- Unit tests
- Stage 1/2/3 (if run)
- Checkpoint averaging
- Inference with self-ensemble + multiscale
- Visualizations
- Drive backup
- TensorBoard

**Next steps / recommendations**
1. Tune `corr_temp`, `proj_dim`, `laplacian_weight` using short fine-tunes and track validation PSNR.
2. For final NTIRE submission: use `swin_depth=4`, `proj_dim=192` if you have 48+ GB GPU or multi-GPU.
3. Average top-k EMA checkpoints by validation PSNR (not just recent).
4. Optionally test small test-time flow refinement on hard frames.
5. Automate hyperparameter sweeps with `submit` jobs or a small optuna-runner.

**If you want:**
- Generate a `.ipynb` file ready to upload
- Add a helper to automatically compute per-frame PSNR/SSIM/LPIPS CSV summary
- Add small scripts to run distributed training (DDP) if you have multi-GPU
